# IMPORTS

In [110]:
import pandas as pd
import numpy as np
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import pkg_resources
from ydata_profiling import ProfileReport
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn import metrics
from sklearn.metrics import roc_curve
from sklearn.metrics import recall_score, confusion_matrix, precision_score, f1_score, accuracy_score, classification_report ,roc_auc_score

from sklearn.model_selection import GridSearchCV , StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier



In [75]:
import sys
print(sys.executable)

d:\customer-churn-prediction\.venv\Scripts\python.exe


In [76]:
df=pd.read_csv(r'D:\customer-churn-prediction\data.csv')

# print(df.head(10))
# df.sample(10)
df.shape


(7043, 21)

In [77]:
df.isnull().sum()
# df.columns.values

df['TotalCharges'] = pd.to_numeric(df.TotalCharges, errors='coerce')
# df.isnull().sum()

# df[df['TotalCharges'].isnull()]
# df[df['tenure']==0]

df.drop(labels=df[df['tenure'] == 0].index, axis=0, inplace=True)
# df[df['tenure'] == 0].index


In [78]:
df = df.drop(['customerID'], axis = 1)
# df.sample(15)

# df[df['TotalCharges'].isnull()]

# df.isnull().sum()

# EDA 

In [79]:
import pkg_resources

from ydata_profiling import ProfileReport
import plotly.express as px


In [80]:
# profile = ProfileReport(
#     df,
#     title="Telecom Customer Churn EDA",
#     minimal=True
# )

# profile.to_file("churn_report.html")

In [81]:
fig = px.pie(
    df,
    names="Churn",
    title="Customer Churn Distribution",
    color="Churn",
    color_discrete_map={
        "Yes": "red",
        "No": "blue"
    },
    hole=0  # 0 creates a pie chart; use 0.4 for a donut chart
)

fig.show()

fig = px.pie(
    df,
    names="PaymentMethod",
    title="Customer Payment Method Distribution",
    hole=0.3
)

fig.update_traces(textinfo="label+percent+value")
fig.show()



In [82]:
df.columns
df.sample(5)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
5363,Male,0,Yes,Yes,70,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.35,1715.15,No
7017,Female,0,No,No,51,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Bank transfer (automatic),20.65,1020.75,No
5275,Male,0,No,No,11,Yes,Yes,DSL,No,No,Yes,No,No,No,Month-to-month,No,Electronic check,53.75,608.00,Yes
3533,Female,0,No,No,10,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.80,198.25,No
2887,Male,0,No,No,1,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,45.05,45.05,Yes


# data preprocessing


In [ ]:
# x and y 

df['Churn']=df['Churn'].map({"Yes":1,"No":0})

df.sample(5)




,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
4219,Female,0,No,No,48,Yes,No,Fiber optic,No,Yes,No,No,No,No,One year,No,Electronic check,75.15,3772.65,0
3367,Male,0,No,No,52,Yes,Yes,DSL,Yes,Yes,Yes,Yes,Yes,Yes,Two year,Yes,Mailed check,89.25,4652.40,0
6368,Male,1,No,No,2,Yes,Yes,Fiber optic,No,No,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,94.00,181.70,1
6129,Male,0,Yes,No,52,No,No phone service,DSL,Yes,No,No,No,No,Yes,Month-to-month,Yes,Electronic check,39.90,2020.90,0
4682,Female,1,No,No,27,Yes,Yes,Fiber optic,No,No,Yes,No,No,Yes,Month-to-month,No,Credit card (automatic),88.30,2467.75,1


In [94]:

X=df.drop(columns='Churn')
Y=df['Churn']

X.columns



Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges'],
      dtype='object')

In [100]:
X_train , X_test , Y_train,Y_test = train_test_split(X,Y,test_size=0.30,random_state=42)

# ENODING THE COLUMNS

In [101]:
numerical_columns=["tenure","MonthlyCharges","TotalCharges"]

#senior citizer already in  0,1 
nominal_columns=["gender","Partner","Dependents","PhoneService","MultipleLines","InternetService","OnlineSecurity","OnlineBackup","DeviceProtection","TechSupport","StreamingTV","StreamingMovies","PaperlessBilling","PaymentMethod"]

ordinal_columns=["Contract"]

In [102]:
preprocessing=ColumnTransformer(
    transformers=[
        ("numerical_columns",StandardScaler(),numerical_columns),
        ("nominal_columns",OneHotEncoder(),nominal_columns),
        ("ordinal_columns",OrdinalEncoder(categories=[["Month-to-month", "One year", "Two year"]]),ordinal_columns)
        ],remainder="passthrough"
)

In [107]:
clf={
    "LogisticRegression":LogisticRegression(max_iter=100,random_state=42),
    "DecisionTreeClassifier":DecisionTreeClassifier(max_depth=5,min_samples_split=10,min_samples_leaf=5),
    "AdaBoostClassifier":AdaBoostClassifier(n_estimators=100,learning_rate=0.1,random_state=42),
    "RandomForestClassifier":RandomForestClassifier(n_estimators=200,max_depth=None,min_samples_leaf=2,n_jobs=-1),
    "GradientBoostingClassifier":GradientBoostingClassifier(n_estimators=100,learning_rate=0.1,random_state=42,max_depth=3)
}

In [108]:
results=[]
for name,clf in clf.items():
    pipe=Pipeline([("preprocess",preprocessing),("clf",clf)])
    pipe.fit(X_train,Y_train)
    cv_scores = cross_val_score(pipe, X_train,Y_train, cv=5, scoring="roc_auc")
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy":  accuracy_score(Y_test, pred),
        "Precision": precision_score(Y_test, pred),
        "Recall":    recall_score(Y_test, pred),
        "F1":        f1_score(Y_test, pred),
        "ROC-AUC":   roc_auc_score(Y_test, proba),
        "CV ROC-AUC":   cv_scores.mean(),
    })

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False).round(3)
print(results_df)

                        Model  Accuracy  Precision  Recall     F1  ROC-AUC  \
4  GradientBoostingClassifier     0.800      0.655   0.528  0.584    0.841   
0          LogisticRegression     0.796      0.640   0.535  0.583    0.840   
2          AdaBoostClassifier     0.787      0.675   0.381  0.487    0.827   
3      RandomForestClassifier     0.787      0.628   0.487  0.548    0.824   
1      DecisionTreeClassifier     0.782      0.587   0.611  0.599    0.821   

   CV ROC-AUC  
4       0.848  
0       0.847  
2       0.843  
3       0.840  
1       0.834  


In [ ]:

### hyper paramters fropm gpt

# 1. Define models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        random_state=42, n_jobs=1
    ),
    "AdaBoost": AdaBoostClassifier(
        random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

# 2. Define hyperparameters for each model
param_grids = {
    "Logistic Regression": {
        "clf__C": [0.01, 0.1, 1, 10],
        "clf__solver": ["lbfgs"]
    },

    "Decision Tree": {
        "clf__max_depth": [3, 5, 10, None],
        "clf__min_samples_split": [2, 10],
        "clf__min_samples_leaf": [1, 5]
    },

    "Random Forest": {
        "clf__n_estimators": [100, 200],
        "clf__max_depth": [None, 10, 20],
        "clf__min_samples_leaf": [1, 2, 5],
        "clf__max_features": ["sqrt", 0.8]
    },

    "AdaBoost": {
        "clf__n_estimators": [50, 100, 200],
        "clf__learning_rate": [0.01, 0.1, 1.0]
    },

    "Gradient Boosting": {
        "clf__n_estimators": [50, 100, 200],
        "clf__learning_rate": [0.01, 0.05, 0.1],
        "clf__max_depth": [1, 2, 3]
    }
}

# 3. Set up stratified 5-fold cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 4. Tune every model
tuning_results = []
best_models = {}

for name, model in models.items():

    print(f"\nTuning {name}...")

    pipe = Pipeline([
        ("preprocess", preprocessing),
        ("clf", model)
    ])

    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grids[name],
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        verbose=1
    )

    # Search using training data only
    search.fit(X_train, Y_train)

    # GridSearchCV refits the best pipeline on all training data
    best_model = search.best_estimator_
    best_models[name] = best_model

    # Evaluate on the separate test set
    pred = best_model.predict(X_test)
    proba = best_model.predict_proba(X_test)[:, 1]

    tuning_results.append({
        "Model": name,
        "Best CV ROC-AUC": search.best_score_,
        "Test Accuracy": accuracy_score(Y_test, pred),
        "Test Precision": precision_score(
            Y_test, pred, zero_division=0
        ),
        "Test Recall": recall_score(
            Y_test, pred, zero_division=0
        ),
        "Test F1": f1_score(
            Y_test, pred, zero_division=0
        ),
        "Test ROC-AUC": roc_auc_score(Y_test, proba),
        "Best Parameters": search.best_params_
    })

    print("Best CV ROC-AUC:", round(search.best_score_, 4))
    print("Best parameters:", search.best_params_)

# 5. Compare tuned models
tuned_results_df = (
    pd.DataFrame(tuning_results)
    .sort_values("Best CV ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

print("\nFinal tuned model comparison:")
print(tuned_results_df.to_string(index=False))



Tuning Logistic Regression...
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best CV ROC-AUC: 0.8465
Best parameters: {'clf__C': 1, 'clf__solver': 'lbfgs'}

Tuning Decision Tree...
Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best CV ROC-AUC: 0.8284
Best parameters: {'clf__max_depth': 5, 'clf__min_samples_leaf': 1, 'clf__min_samples_split': 10}

Tuning Random Forest...
Fitting 5 folds for each of 36 candidates, totalling 180 fits
Best CV ROC-AUC: 0.8463
Best parameters: {'clf__max_depth': 10, 'clf__max_features': 'sqrt', 'clf__min_samples_leaf': 5, 'clf__n_estimators': 200}

Tuning AdaBoost...
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best CV ROC-AUC: 0.8486
Best parameters: {'clf__learning_rate': 1.0, 'clf__n_estimators': 200}

Tuning Gradient Boosting...
Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best CV ROC-AUC: 0.8509
Best parameters: {'clf__learning_rate': 0.1, 'clf__max_depth': 1, 'clf__n_estimators': 200}

Final 

In [ ]:

## hyper pram using claude

search_space = {
    "Logistic Regression": (
        LogisticRegression(max_iter=1000),
        {"clf__C": [0.01, 0.1, 1, 10], "clf__class_weight": [None, "balanced"]},
    ),
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {"clf__max_depth": [3, 5, 8], "clf__min_samples_leaf": [1, 5, 20]},
    ),
    "Random Forest": (
        RandomForestClassifier(random_state=42),
        {"clf__n_estimators": [100, 300], "clf__max_depth": [5, 10, None],
         "clf__min_samples_leaf": [1, 5]},
    ),
    "AdaBoost": (
        AdaBoostClassifier(random_state=42),
        {"clf__n_estimators": [50, 100, 200], "clf__learning_rate": [0.01, 0.1, 1]},
    ),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=42),
        {"clf__n_estimators": [100, 200], "clf__learning_rate": [0.03, 0.1],
         "clf__max_depth": [2, 3, 4]},
    ),
}

best = {}
for name, (clf, params) in search_space.items():
    pipe = Pipeline([("preprocess", preprocessing), ("clf", clf)])
    gs = GridSearchCV(pipe, params, cv=5, scoring="roc_auc", n_jobs=-1)
    gs.fit(X_train, Y_train)
    best[name] = gs
    print(f"{name}: CV AUC={gs.best_score_:.3f} | {gs.best_params_}")


from sklearn.metrics import roc_auc_score
for name, gs in best.items():
    auc = roc_auc_score(Y_test, gs.predict_proba(X_test)[:, 1])
    print(f"{name}: test AUC={auc:.3f}")

Logistic Regression: CV AUC=0.847 | {'clf__C': 10, 'clf__class_weight': 'balanced'}
Decision Tree: CV AUC=0.834 | {'clf__max_depth': 5, 'clf__min_samples_leaf': 1}
Random Forest: CV AUC=0.849 | {'clf__max_depth': 10, 'clf__min_samples_leaf': 5, 'clf__n_estimators': 300}
AdaBoost: CV AUC=0.849 | {'clf__learning_rate': 1, 'clf__n_estimators': 200}
Gradient Boosting: CV AUC=0.851 | {'clf__learning_rate': 0.03, 'clf__max_depth': 2, 'clf__n_estimators': 200}
Logistic Regression: test AUC=0.839
Decision Tree: test AUC=0.818
Random Forest: test AUC=0.836
AdaBoost: test AUC=0.841
Gradient Boosting: test AUC=0.839
